# Gene Searcher v1.0

In [14]:
import os
import json
import sqlite3
import urllib.parse
import requests
import openai
from bs4 import BeautifulSoup
# from pydantic import BaseModel, Field as Field_
import pydantic as pyd
from typing import Optional
from dotenv import load_dotenv
from rich import print


from sqlmodel import Field, Session, SQLModel, create_engine, select
from duckduckgo_search import DDGS

In [ ]:


results = DDGS().text("FASN hypomethylated cancer", max_results=5)
print(results)


[
    {
        'title': 'FASN Gene Methylation is Associated with Fatty Acid Synthase Expression ...',
        'href': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10795515/',
        'body': 'FASN protein is upregulated and FASN gene is hypomethylated in tumor compared with benign tissue 
in JHU primary tumor cohort.A, Dual FASN and p63 IHC assay in representative primary prostate tumor from JHU cohort
(scale bar = 250 µm). FASN and p63 are both labeled in brown. Benign glands with nuclear p63 labeling in basal 
cells (arrow) are then manually excluded from analysis and FASN scoring ...'
    },
    {
        'title': 'Hypomethylated gene RAC3 induces cell proliferation and invasion by ...',
        'href': 'https://www.sciencedirect.com/science/article/pii/S1357272522001194',
        'body': 'In ovarian cancer, FASN impairs the anti-tumor immune response through lipid accumulation in 
tumor-infiltrating DCs and subsequently induces T cell exclusion and dysfunction (Jiang et al., 2018). In EC, FASN 
was overexpressed in EC tissues ( Anagnostou et al., 2017 ), and long noncoding RNA SNHG25 promoted EC progression 
via upregulation of ...'
    },
    {
        'title': 'Comprehensive Analysis of FASN in Tumor Immune Infiltration and ...',
        'href': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9779179/',
        'body': 'In addition, we examined the FASN mutation status in all types of cancer and found that FASN 
showed a higher gene mutation frequency in SKCM (13.52%), UCEC (13.24%), and LIHC (10.49%) (Supplementary Figure 
S3A). Immune infiltration in LIHC, SKCM, and UCEC was significantly influenced by somatic copy number variation 
(Supplementary Figure S3B).'
    },
    {
        'title': 'Hypomethylated gene RAC3 induces cell proliferation and invasion by ...',
        'href': 'https://www.sciencedirect.com/science/article/abs/pii/S1357272522001194',
        'body': 'Hypomethylated gene RAC3 induces cell proliferation and invasion by increasing FASN expression in 
endometrial cancer. ... Endometrial cancer (EC) is one of the most common malignancies in women and accounts for 
more than 76,000 deaths among women worldwide each year (Urick and Bell, 2019). ... (Qiu et al., 2013). The FASN 
signaling pathway is ...'
    },
    {
        'title': 'FASN Gene Methylation is Associated with Fatty Acid Synthase ... - PubMed',
        'href': 'https://pubmed.ncbi.nlm.nih.gov/38112617/',
        'body': 'We demonstrate that the FASN gene body is hypomethylated and overexpressed in primary prostate 
tumors compared to benign tissue, and FASN gene methylation is significantly inversely correlated with FASN protein
or gene expression in both primary and metastatic prostate cancer.'
    }
]

In ovarian cancer, FASN impairs the anti-tumor immune response through lipid accumulation in tumor-infiltrating DCs
and subsequently induces T cell exclusion and dysfunction (Jiang et al., 2018). In EC, FASN was overexpressed in EC
tissues ( Anagnostou et al., 2017 ), and long noncoding RNA SNHG25 promoted EC progression via upregulation of ...

In [4]:
load_dotenv()
# Set your OpenAI API key
openai.api_key = os.getenv("OPENAI_API_KEY")  # or directly assign your key string


In [27]:




class GeneMethylationInfo(pyd.BaseModel):
    Gene_name: str = pyd.Field(None, description="The name of the gene")
    methylation_status: Optional[str] = pyd.Field(
        None, description="Hypermethylated or Hypomethylated"
    )
    phenotype: Optional[str] = pyd.Field(
        None, description="Phenotypic description related to methylation"
    )
    gene_function: Optional[str] = pyd.Field(
        None, description="Function of the gene in the cell"
    )
    role_in_cancer: Optional[str] = pyd.Field(
        None, description="Role of the gene in cancer"
    )
    Disease: Optional[str] = pyd.Field(None, description="Associated disease(s)")
    url: str = pyd.Field(None, description="URL of the source")


# -------------------------------
# Database (SQLite) Setup Functions
# -------------------------------
DB_FILENAME = "gene_methylation.db"
sqlite_url = f"sqlite:///{DB_FILENAME}"
engine = create_engine(sqlite_url)

class GeneTable(SQLModel, table=True):
    __tablename__ = "gene_methylation"
    
    gene: str = Field(primary_key=True)
    methylation_status: Optional[str] = None
    phenotype: Optional[str] = None
    role_in_cancer: Optional[str] = None
    disease: Optional[str] = None

def init_db():
    SQLModel.metadata.create_all(engine)


def get_cached_gene_info(gene: str) -> Optional[GeneMethylationInfo]:
    with Session(engine) as session:
        statement = select(GeneTable).where(GeneTable.gene == gene)
        result = session.exec(statement).first()
        if result:
            return GeneMethylationInfo(
                Gene_name=result.gene,
                methylation_status=result.methylation_status,
                phenotype=result.phenotype,
                role_in_cancer=result.role_in_cancer,
                Disease=result.disease
            )
    return None

def save_gene_info_to_cache(gene_info: GeneMethylationInfo):
    with Session(engine) as session:
        gene_record = GeneTable(
            gene=gene_info.Gene_name,
            methylation_status=gene_info.methylation_status,
            phenotype=gene_info.phenotype,
            role_in_cancer=gene_info.role_in_cancer,
            disease=gene_info.Disease
        )
        session.merge(gene_record)
        session.commit()


def search_gene_info(gene: str, methylation_status: str, search_query: str, max_results: int = 5) -> list:
    query = f"{gene} {methylation_status} {search_query} cancer"
    results = DDGS().text(query, max_results=max_results)
    return results

def find_best_results(results: list[dict[str, str]]) -> str:
    """Use small OpenAI to find best result."""
    # Combine all search results into a single text
    combined_text = ""
    for result in results:
        title = result.get('title', '')
        body = result.get('body', '')
        combined_text += f"Title: {title}\nContent: {body}\n\n url: {url}\n\n"
    
    if not combined_text:
        return "No relevant information found."
    
    prompt = f"""
    Analyze the following search results and extract relevant information about gene methylation, 
    its role in cancer, and associated phenotypes. Focus on methylation status and disease relationships:

    {combined_text}

    Please provide a concise summary of the most relevant findings. and associated phenotypes. Focus on methylation status and disease relationships.
    
    """
    
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a scientific literature analyzer focusing on gene methylation in cancer."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
            max_tokens=2000,

            
            
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error in processing search results: {e}")
        return "Error processing search results."



def extract_methylation_info_with_openai(text: str, gene: str) -> GeneMethylationInfo:
    """
    Uses the OpenAI API to extract methylation information.
    The prompt instructs the model to return a JSON object with the keys:
    Gene_name, methylation_status, phenotype, role_in_cancer, Disease.
    """
    prompt = f"""
    You are an expert in molecular biology and cancer research. Given the following text, extract the following information about the gene "{gene}":
    """ 
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You extract specific scientific data from text."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=500,
        result_type=GeneMethylationInfo
    )
    return response.choices[0].message.content.strip()



        


/Users/priyesh.rughani/miniconda3/envs/ai_tools/lib/python3.12/site-packages/sqlmodel/main.py:638: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.GeneTable, and will be replaced in the string-lookup table.
  DeclarativeMeta.__init__(cls, classname, bases, dict_, **kw)


InvalidRequestError: Table 'gene_methylation' is already defined for this MetaData instance.  Specify 'extend_existing=True' to redefine options and columns on an existing Table object.

In [28]:
def run_workflow():
    # Initialize the database (creates the table if it doesn't exist)
    init_db()

    genes_to_analyze = ["FASN", "TP53", "BRCA1"]
    methylated_status = ["hypermethylated", "methylated", "hypermethylated"]

    for gene, methyl_status in zip(genes_to_analyze, methylated_status):
        # Check if the gene's information is already cached.
        cached_info = get_cached_gene_info(gene)
        if cached_info:
            print(f"Returning cached information for {gene}.")
            print(cached_info)
            continue

        # Search for relevant URLs.
        urls = search_gene_info(gene, methyl_status, "cancer", max_results=5)
        print(urls)
        result = find_best_results(urls)
        print(result)
        final_output = extract_methylation_info_with_openai(result, gene)
        print(final_output)
        
        

run_workflow()

[
    {
        'title': 'Comprehensive Analysis of FASN in Tumor Immune Infiltration and ...',
        'href': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9779179/',
        'body': 'FASN expression profiles in normal tissues and cancers. (A) FASN expression levels in normal 
tissues and organs. (B) Profiles of FASN expression in 33 cancer types. (C) ... while the gene body is 
hypermethylated. The methylation level of the probe (TSS200: cg24715260) was significantly negatively correlated 
with the expression of FASN in a ...'
    },
    {
        'title': 'Fatty acid synthase (FASN) is a tumor-cell-intrinsic metabolic ...',
        'href': 'https://www.nature.com/articles/s41420-024-02184-z',
        'body': 'Reactivation of de novo lipogenesis catalyzed by fatty acid synthase (FASN) is a nearly universal,
targetable hallmark of the metabolic remodeling that occurs in cancer cells [1,2,3,4,5,6].The ...'
    },
    {
        'title': 'FASN Gene Methylation is Associated with Fatty Acid Synthase Expression ...',
        'href': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10795515/',
        'body': 'Association of FASN Expression and Methylation with ERG Status in Primary Prostate Cancer. FASN 
protein expression has previously been shown to be increased in prostate tumors ... gene fusions. Similar findings 
have been reported for numerous other genes, such as HOXB13 or genes that are hypermethylated and underexpressed in
tumor ...'
    },
    {
        'title': 'Comprehensive Analysis of FASN in Tumor Immune Infiltration and ...',
        'href': 'https://pubmed.ncbi.nlm.nih.gov/36555243/',
        'body': 'Fatty acid synthase (FASN) promotes tumor progression in multiple cancers. In this study, we 
comprehensively examined the expression, prognostic significance, and promoter methylation of FASN, and its 
correlation with immune cell infiltration in pan-cancer. Our results demonstrated that elevated FAS …'
    },
    {
        'title': 'Involvement of the pro-oncogenic enzyme fatty acid synthase in the ...',
        'href': 'https://www.nature.com/articles/s41389-023-00460-8',
        'body': 'An accelerated de novo lipogenesis (DNL) flux is a common characteristic of cancer cells required 
to sustain a high proliferation rate. The DNL enzyme fatty acid synthase (FASN) is overexpressed ...'
    }
]

Error in processing search results: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at 
https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742

Error processing search results.

Error extracting information for gene FASN: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at 
https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742

GeneMethylationInfo(
    Gene_name='FASN',
    methylation_status=None,
    phenotype=None,
    gene_function=None,
    role_in_cancer=None,
    Disease=None,
    url=None
)

...

[
    {
        'title': 'BRCA1 Promoter Hypermethylation in Malignant Breast Tumors and in the ...',
        'href': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10530732/',
        'body': 'The hypermethylation status of the promoter region of the breast cancer 1 (BRCA1), a well-known 
tumor suppressor gene, has been extensively investigated in the last two decades as a potential biomarker for 
breast cancer. ... The presence of a tubule formation score of 3 was more frequent in the BRCA1 hypermethylated 
group (15/17; 88.24% ...'
    },
    {
        'title': 'Comprehensive molecular comparison of BRCA1 hypermethylated and BRCA1 ...',
        'href': 'https://www.nature.com/articles/s41467-020-17537-2',
        'body': 'BRCA1 hypermethylated cases share, with few exceptions, similar frequencies of copy number 
alterations, proportions of reported driver genes in breast cancer, tumor mutational burden, mutational ...'
    },
    {
        'title': 'Comprehensive molecular comparison of BRCA1 hypermethylated and BRCA1 ...',
        'href': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7385112/',
        'body': 'BRCA1 hypermethylated cases share, with few exceptions, similar frequencies of copy number 
alterations, proportions of reported driver genes in breast cancer, tumor mutational burden, mutational signatures,
rearrangement signatures, gene expression patterns, immune cell infiltration, and DNA methylation patterns with 
BRCA1-null cases.'
    },
    {
        'title': 'BRCA1 and Breast Cancer: a Review of the Underlying Mechanisms ...',
        'href': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6438831/',
        'body': 'Abstract. Since the first cloning of BRCA1 in 1994, many of its cellular interactions have been 
elucidated. However, its highly specific role in tumorigenesis in the breast tissue—carriers of BRCA1 mutations are
predisposed to life-time risks of up to 80%—relative to many other tissues that remain unaffected, has not yet been
fully enlightened. In this article, we have applied a universal ...'
    },
    {
        'title': 'Constitutional BRCA1 Methylation and Risk of Incident Triple-Negative ...',
        'href': 'https://jamanetwork.com/journals/jamaoncology/fullarticle/2796099',
        'body': 'Women with breast cancer type 1 susceptibility gene (BRCA1) germline pathogenic variants are at 
high risk of triple-negative breast cancer (TNBC) and high-grade serous ovarian cancer (HGSOC). 1 \ufeff BRCA1, 
like BRCA2 and several other genes, is required for homologous recombination repair of double-stranded DNA breaks. 
2 While most TNBCs and ...'
    }
]

Error in processing search results: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at 
https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742

Error processing search results.

Error extracting information for gene BRCA1: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at 
https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742

GeneMethylationInfo(
    Gene_name='BRCA1',
    methylation_status=None,
    phenotype=None,
    gene_function=None,
    role_in_cancer=None,
    Disease=None,
    url=None
)

In [26]:
# url = "https://pmc.ncbi.nlm.nih.gov/articles/PMC9779179/"
url = "https://pmc.ncbi.nlm.nih.gov/articles/PMC9779179/"

# bs4 = BeautifulSoup(requests.get(url).text, "html.parser")
# print(bs4.title.text)
# print(bs4.get_text())

response = requests.get(url, timeout=5, headers={"User-Agent": "Mozilla/5.0"})
print(response.status_code)
if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")
    page_text = soup.get_text(separator=" ", strip=True)
    print(page_text)
    


200

Comprehensive Analysis of FASN in Tumor Immune Infiltration and Prognostic Value for Immunotherapy and Promoter DNA
Methylation - PMC Skip to main content An official website of the United States government Here's how you know 
Here's how you know Official websites use .gov A .gov website belongs to an official
                            government organization in the United States. Secure .gov websites use HTTPS A lock ( 
Lock Locked padlock icon ) or https:// means you've safely
                                connected to the .gov website. Share sensitive
                                information only on official, secure websites. Search Log in Dashboard Publications
Account settings Log out Search… Search NCBI Primary site navigation Search Logged in as: Dashboard Publications 
Account settings Log in Search PMC Full-Text Archive Search in PMC Advanced Search Journal List User Guide 
PERMALINK Copy As a library, NLM provides access to scientific literature. Inclusion in an NLM database does not 
imply endorsement of, or agreement with,
    the contents by NLM or the National Institutes of Health. Learn more: PMC Disclaimer | PMC Copyright Notice Int
J Mol Sci . 2022 Dec 9;23(24):15603. doi: 10.3390/ijms232415603 Search in PMC Search in PubMed View in NLM Catalog 
Add to search Comprehensive Analysis of FASN in Tumor Immune Infiltration and Prognostic Value for Immunotherapy 
and Promoter DNA Methylation Mingyang Zhang Mingyang Zhang 1 School of Life Science, Inner Mongolia University, 
Hohhot 010020, China Conceptualization, Investigation Find articles by Mingyang Zhang 1, † , Lei Yu Lei Yu 1 School
of Life Science, Inner Mongolia University, Hohhot 010020, China Software, Investigation Find articles by Lei Yu 1,
† , Yannan Sun Yannan Sun 1 School of Life Science, Inner Mongolia University, Hohhot 010020, China Methodology, 
Investigation Find articles by Yannan Sun 1 , Li Hao Li Hao 1 School of Life Science, Inner Mongolia University, 
Hohhot 010020, China Validation, Investigation Find articles by Li Hao 1 , Jing Bai Jing Bai 1 School of Life 
Science, Inner Mongolia University, Hohhot 010020, China Validation, Investigation Find articles by Jing Bai 1 , 
Xinyu Yuan Xinyu Yuan 1 School of Life Science, Inner Mongolia University, Hohhot 010020, China Investigation Find 
articles by Xinyu Yuan 1 , Rihan Wu Rihan Wu 1 School of Life Science, Inner Mongolia University, Hohhot 010020, 
China Data curation Find articles by Rihan Wu 1 , Mei Hong Mei Hong 1 School of Life Science, Inner Mongolia 
University, Hohhot 010020, China Writing – review & editing Find articles by Mei Hong 1 , Pengxia Liu Pengxia Liu 1
School of Life Science, Inner Mongolia University, Hohhot 010020, China Writing – review & editing Find articles by
Pengxia Liu 1 , Xiaojun Duan Xiaojun Duan 1 School of Life Science, Inner Mongolia University, Hohhot 010020, China
Find articles by Xiaojun Duan 1 , Changshan Wang Changshan Wang 1 School of Life Science, Inner Mongolia 
University, Hohhot 010020, China Conceptualization Find articles by Changshan Wang 1, * Editors: Venkatesh Hegde 1 
, Stephanie M Dorta-Estremera 1 Author information Article notes Copyright and License information 1 School of Life
Science, Inner Mongolia University, Hohhot 010020, China * Correspondence: changshanwang@imu.edu.cn † These authors
contributed equally to this work. Roles Mingyang Zhang : Conceptualization, Investigation Lei Yu : Software, 
Investigation Yannan Sun : Methodology, Investigation Li Hao : Validation, Investigation Jing Bai : Validation, 
Investigation Xinyu Yuan : Investigation Rihan Wu : Data curation Mei Hong : Writing – review & editing Pengxia Liu
: Writing – review & editing Changshan Wang : Conceptualization Venkatesh Hegde : Academic Editor Stephanie M 
Dorta-Estremera : Academic Editor Received 2022 Oct 31; Accepted 2022 Dec 1; Collection date 2022 Dec. © 2022 by 
the authors. Licensee MDPI, Basel, Switzerland. This article is an open access article dist